In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from transformers import PreTrainedTokenizerFast

from torch.optim.lr_scheduler import LambdaLR


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

Using device: cuda


## 1. Prepare Training Data

We use simple English sentences for token-level modeling, with a standard HuggingFace tokenizer.

In [14]:
VOCAB_SIZE = 10000
# NUM_EPOCHS = 30
NUM_EPOCHS = 100

In [15]:
# Read training corpus
with open("comm1-7.txt", "r", encoding="utf-8") as f:
    training_text = f.read()

print(f"Training corpus length: {len(training_text)} characters")
print(f"First 100 chars: {training_text[:100]}")

# Create a custom tokenizer with a small vocabulary
# BPE: Byte Pair Encoding
# ByteLevel: allows handling of latin and UTF-8 characters;
#   it represents each character as sequence of bytes.
#   For instance "é" is represented as two bytes: 0xc3 0xa9.
#   The tokenizer will learn to combine these bytes into meaningful tokens.
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()
tokenizer.decoder = ByteLevelDecoder()

# Train the tokenizer with a limited vocabulary size
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    min_frequency=2
)

# Save text to temporary file for training
with open("temp_train.txt", "w", encoding="utf-8") as f:
    f.write(training_text)

tokenizer.train(["temp_train.txt"], trainer)

# Wrap in HuggingFace tokenizer for convenience
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer)
tokenizer.pad_token = "[PAD]"

print(f"\nTokenizer vocabulary size: {len(tokenizer)}")
print(f"Sample tokens: {list(tokenizer.get_vocab().keys())[:20]}")

Training corpus length: 35655 characters
First 100 chars: LA DIVINA COMMEDIA
di Dante Alighieri
INFERNO



Inferno: Canto I

  Nel mezzo del cammin di nostra 

Tokenizer vocabulary size: 1684
Sample tokens: ['Ġconoscer', 'Ġpiedi', 'Ġsmarrito', 'Ġdo', 'ĠBeatrice', 'degna', 'Ã¹', 'Ġvoler', 'ni', 'altro', 'ĠP', 'ola', 'Ġrabb', 'Ġdue', 'atÃł', 'cedi', 'Ġdiparti', '[PAD]', 'petto', 'Ġlargo']


In [16]:
# Build token vocabulary using the tokenizer
vocab_size = tokenizer.vocab_size

# Helper functions for encoding/decoding
def encode(text: str) -> list:
    # pt -> PyTorch tensors
    # [0] to get the first (only) sequence from the batch
    return tokenizer(text, return_tensors="pt")["input_ids"][0]

def decode(ids: list) -> str:
    # Use the tokenizer's decode method with skip_special_tokens
    # This properly handles subword merging
    # skip_sepcial_tokens=True removes tokens like [PAD], [CLS], etc.
    return tokenizer.decode(ids, skip_special_tokens=True)


input_ids: torch.Tensor = encode(training_text)

# Test encoding/decoding
sample = "Nel mezzo del cammin di nostra vita"
encoded = encode(sample)
decoded = decode(encoded)
print(f"\nOriginal: {sample}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")


Original: Nel mezzo del cammin di nostra vita
Encoded: tensor([ 305,   93, 1401,  190, 1593,   83,  733,  475])
Decoded:  Nel mezzo del cammin di nostra vita


## 3. Create Dataset for Sequence Generation

For autoregressive generation, each training example is a sequence of tokens where we predict the next token.

In [17]:
SEQ_LEN = 32  # Context window size (in tokens)

class TokenDataset(Dataset):
    # seq_len: length of the context window
    # A songle sequence of tokens generates multiple samples by sliding the window
    def __init__(self, input_ids: torch.Tensor, seq_len: int):
        self.data = input_ids
        self.seq_len = seq_len

    def __len__(self):
        # Number of samples is total tokens minus the context window size
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        # Input: tokens from idx to idx+seq_len
        # Target: tokens from idx+1 to idx+seq_len+1 (shifted by 1)
        if idx < 0 or idx >= len(self):
            raise IndexError(f"Index {idx} out of range [0, {len(self)}]")
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y

dataset = TokenDataset(input_ids, SEQ_LEN)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(loader)}")

# example
x_sample, y_sample = dataset[1000]
print(f"\nExample input:  {decode(x_sample.tolist())}")
print(f"Example target: {decode(y_sample.tolist())}")

token_strings = tokenizer.convert_ids_to_tokens(x_sample.tolist())
print(token_strings)


Dataset size: 12331
Number of batches: 771

Example input:   ritorni a tanta noia?
perché non sali il dilettoso monte
ch'è principio e cagion di tutta gioia?".
  "Or
Example target: ni a tanta noia?
perché non sali il dilettoso monte
ch'è principio e cagion di tutta gioia?".
  "Or se
['Ġritor', 'ni', 'Ġa', 'Ġtanta', 'Ġno', 'ia', '?', 'Ċ', 'per', 'chÃ©', 'Ġnon', 'Ġsali', 'Ġil', 'Ġdiletto', 'so', 'Ġmonte', 'Ċ', 'ch', "'", 'Ã¨', 'Ġprincipio', 'Ġe', 'Ġcagion', 'Ġdi', 'Ġtutta', 'Ġgio', 'ia', '?".', 'Ċ', 'Ġ', 'Ġ"', 'Or']


### 4. Build Transformer Decoder for Generation

### Positional Encoding
Torch transformers do not define positional encodings internally, so we need to implement them ourselves.

In [18]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        # unsqueeze(1) adds a dimension, making position shape (max_len, 1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # div_term, vector with shape (d_model/2,)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model)
        )
        # position * div_term results in shape (max_len, d_model/2) due to broadcasting
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Register pe as a buffer to avoid it being considered a model parameter
        # it will not be updated during training, but will be saved/loaded with the model
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[: x.size(1)]



In [19]:
class TransformerGenerator(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        nhead: int = 4,
        num_layers: int = 3,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model
        # an Embedding layer is basically a lookup table that maps
        # token IDs to dense vectors. It is updated during the
        # backpropagation step with the updated values for each
        # involved token.
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        # defines a single decoder layer
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        # stack multiple decoder layers
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers)

        self.output_layer = nn.Linear(d_model, vocab_size)

    def generate_causal_mask(self, sz: int) -> torch.Tensor:
        """Generate causal mask to prevent attending to future positions."""
        # triu: upper triangular part of a matrix
        # diagonal=1 ensures that the main diagonal is zeroed out (only diagonals
        # that lie above the main of at least 1 position are set to True)
        mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
        return mask

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, seq_len)
        seq_len = x.size(1)

        # Create causal mask
        causal_mask = self.generate_causal_mask(seq_len).to(x.device)

        # Embed and add positional encoding
        # Scale embeddings by sqrt(d_model) as per Transformer paper
        # in this way the range of the embeddings is comparable to the
        # range of positional encodings.
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # tgt: the input to the decoder
        # memory: in theory should contain the encoder outputs. In decoder-only
        # architectures, it is the same as the target.
        # tgt_mask: prevents attending to future positions from the target
        # memory_mask: prevents attending to future positions from the memory
        x = self.transformer_decoder(
            tgt=x,
            memory=x,
            tgt_mask=causal_mask,
            memory_mask=causal_mask,
        )

        # Project to vocabulary
        logits = self.output_layer(x)
        return logits


model = TransformerGenerator(vocab_size=vocab_size).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 1,226,516


### 5. Training

In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Learning rate warmup schedule
num_epochs = NUM_EPOCHS
num_batches_per_epoch = len(loader)
total_steps = num_epochs * num_batches_per_epoch
warmup_steps = int(0.1 * total_steps)  # 10% of total steps for warmup

# It returns a multiplicative factor for the initial lr.
# Up until warmup_steps, the learning rate increases linearly from 0 to the initial lr.
# After warmup, it decreases linearly back to 0.
def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    return max(0.0, float(total_steps - current_step) / float(max(1, total_steps - warmup_steps)))

scheduler = LambdaLR(optimizer, lr_lambda)

print("Training...")
global_step = 0
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)

        # Reshape for loss computation
        # logits: (batch, seq_len, vocab_size) -> (batch * seq_len, vocab_size)
        # targets: (batch, seq_len) -> (batch * seq_len)
        # This is the correct shape since CrossEntropyLoss expects
        # inputs of shape (N, C) and targets of shape (N,)
        loss = criterion(logits.view(-1, vocab_size), batch_y.view(-1))

        loss.backward()
        # update gradients = gradients / ||gradients|| if the norm exceeds 1.0
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        global_step += 1

    avg_loss = total_loss / len(loader)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch + 1}: loss={avg_loss:.3f}, lr={current_lr:.2e}")

Training...
Epoch 1: loss=6.588, lr=1.00e-04
Epoch 2: loss=5.678, lr=2.00e-04
Epoch 3: loss=5.305, lr=3.00e-04
Epoch 4: loss=4.831, lr=4.00e-04
Epoch 5: loss=4.226, lr=5.00e-04
Epoch 6: loss=3.563, lr=6.00e-04
Epoch 7: loss=2.947, lr=7.00e-04
Epoch 8: loss=2.407, lr=8.00e-04
Epoch 9: loss=1.973, lr=9.00e-04
Epoch 10: loss=1.647, lr=1.00e-03
Epoch 11: loss=1.354, lr=9.89e-04
Epoch 12: loss=1.123, lr=9.78e-04
Epoch 13: loss=0.947, lr=9.67e-04
Epoch 14: loss=0.828, lr=9.56e-04
Epoch 15: loss=0.739, lr=9.44e-04
Epoch 16: loss=0.672, lr=9.33e-04
Epoch 17: loss=0.626, lr=9.22e-04
Epoch 18: loss=0.589, lr=9.11e-04
Epoch 19: loss=0.552, lr=9.00e-04
Epoch 20: loss=0.523, lr=8.89e-04
Epoch 21: loss=0.499, lr=8.78e-04
Epoch 22: loss=0.476, lr=8.67e-04
Epoch 23: loss=0.460, lr=8.56e-04
Epoch 24: loss=0.442, lr=8.44e-04
Epoch 25: loss=0.428, lr=8.33e-04
Epoch 26: loss=0.416, lr=8.22e-04
Epoch 27: loss=0.404, lr=8.11e-04
Epoch 28: loss=0.393, lr=8.00e-04
Epoch 29: loss=0.379, lr=7.89e-04
Epoch 30: l

### 6. Generazione

In [22]:
def generate_text(
    model: nn.Module,
    start_text: str,
    max_length: int = 100,
    temperature: float = 1.0,
) -> str:
    """
    Generate text autoregressively (token-wise).
    Args:
        model: Trained transformer model
        start_text: Initial text to start generation
        max_length: Maximum number of tokens to generate
        temperature: Sampling temperature (higher = more random)
    """
    model.eval()

    # Encode starting text
    context = encode(start_text)
    generated = context.tolist()  # Convert tensor to list

    with torch.no_grad():
        for _ in range(max_length):
            # Take last SEQ_LEN tokens as context
            context_window = generated[-SEQ_LEN:]
            x = torch.tensor([context_window], dtype=torch.long).to(device)

            # Get predictions
            logits = model(x)

            # Get logits for the last position
            # this is: first (and only) batch, last token in sequence, all vocab
            logits = logits[0, -1, :] / temperature

            # Sample from distribution
            probs = torch.softmax(logits, dim=0)
            next_id = torch.multinomial(probs, num_samples=1).item()

            # Append to generated sequence
            generated.append(next_id)

    return decode(generated)

# Generate text with different starting prompts
print("\nGenerated Text Examples (token-wise):\n")

prompts = ["Nel", "vuolsi", "vate", "spaura"]

for prompt in prompts:
    generated = generate_text(model, prompt, max_length=80, temperature=1.4)
    print(f"Prompt: '{prompt}'")
    print(f"Generated: {generated}")
    print()


Generated Text Examples (token-wise):

Prompt: 'Nel'
Generated:  Nel mezzo del cammin di nostra vita
mi ritrovai per una selva oscura
ché la diritta via era smarrita.
  Ahi quanto a dir qual era è cosa dura
esta selva selvaggia e aspra e forte
che nel pensier rinova la paura!
  Tant'è amara che poco è più morte;
ma per trattar del ben ch'

Prompt: 'vuolsi'
Generated:  vuolsi e con la traiù non mi 'l pelanchi;
l modo ancor ch' tua è 'l terzo, e l'ultimo Lucano.
  Però che ciascun meco si convene
nel nome che sonò la voce sola,
fannomi onore, e di ciò fanno bene".
  Così vid'i' adunar la bella

Prompt: 'vate'
Generated:  vate,
questi chi son c'hanno cotanta onranza,
che dal modo de li altri li diparte?".
  E quelli a me: "L'onrata nominanza
che di lor suona sù ne la tua vita,
grazia acquista in ciel che sì li avanza".
  Intanto voce fu per me udita:
"Onorate

Prompt: 'spaura'
Generated:  spaura che trema.
  E vegno in parte ove non è che luca.



Inferno: Canto V

  Così discesi del cer